In [3]:
# lectura de los datos

import pandas as pd
import read_data

# Suponiendo que el activo tiene ese nombre
# solo es remplazar por el nombre que se tenga
# solo se acepta archivos .parquet y .csv
nombre_activo: str = "EURUSD.parquet"
data: pd.DataFrame = read_data.read_asset(nombre_activo)

primer_lunes = data[data.index.dayofweek == 0].index[0]

data = data[primer_lunes:]

In [ ]:
import find_best
import keys
import pandas as pd
from use_tecnics import simple_methods

keys.candles = 20
keys.methods = simple_methods

ventana_entrenamiento = pd.Timedelta(weeks=3) + pd.Timedelta(days=4)
inicio_testeo = pd.Timedelta(days=3)
fin_testeo = pd.Timedelta(days=4)

inicio_train = data.index[0].normalize()
fin_datos = data.index[-1]

while True:
    train_time = inicio_train + ventana_entrenamiento
    inicio_test = train_time + inicio_testeo
    fin_test = inicio_test + fin_testeo
    print(f"Entrenamiento: {inicio_train.strftime('%Y-%m-%d')} a {train_time.strftime('%Y-%m-%d')}")

    sub_data = data[inicio_train: train_time]
    best_ma = find_best.opti_main(sub_data)

    inicio_train = inicio_train + pd.Timedelta(days=7)
    if train_time >= fin_datos:
        break

    minutos_de_colchon = best_ma[1]*best_ma[2]*(1 if best_ma[0] in simple_methods else 8)
    minutos_de_colchon = train_time - pd.Timedelta(minutes=minutos_de_colchon)
    sub_data = pd.concat([sub_data[minutos_de_colchon:],
                          data[inicio_test: fin_test]])

    print(f"\nTesteo: {inicio_test.strftime('%Y-%m-%d')} a {fin_test.strftime('%Y-%m-%d')}")
    find_best.read_results(best_ma, sub_data)

Entrenamiento: 2026-02-02 a 2026-02-27
Resultado obtenido entrenando desde 2026-02-02 hasta 2026-02-26
Método: WMA, Datos optimizados [np.int64(20), np.int64(108)]

hit ratio: 0.5681818181818182
risk reward: 0.9425507515884292
profit factor: 1.240198357353196
trades: 88
Resultado de sobre ajuste -0.19028261492282803

Testeo: 2026-03-02 a 2026-03-06
Rendimiento de una ma con:
método: WMA, vela: 20, añadidos: [np.int64(108)]
hit ratio: 0.45
risk reward: 0.7183188491186249
profit factor: 0.5877154220061477
trades: 20


Entrenamiento: 2026-02-09 a 2026-03-06
Resultado obtenido entrenando desde 2026-02-09 hasta 2026-03-05
Método: KAMA, Datos optimizados [np.int64(15), np.int64(67)]

hit ratio: 0.6746031746031746
risk reward: 0.8181868887073992
profit factor: 1.696241110734852
trades: 126
Resultado de sobre ajuste 1.5385905137078384

Testeo: 2026-03-09 a 2026-03-13
Rendimiento de una ma con:
método: KAMA, vela: 15, añadidos: [np.int64(67)]
hit ratio: 0.0
risk reward: 0.0
profit factor: 0.0
t